# Entering the Algorithm Prime Triad

AuthorityEffect: None. This notebook exercises the Algorithm Prime Triad (Prime 000, 001, 002) runtime. Every number is computed exactly over the rationals by the package code.

In [ ]:
import sys
from fractions import Fraction
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "oe_prime_algorithms.md").is_file())
sys.path.insert(0, str(root / "code"))
print("package:", root.name)

import oe_prime_algorithms as opa
print("Algorithm Prime module loaded successfully")

## 1. Prime 000: Admission Precedes Measurement

In Prime 000, candidate intake occurs in two phases: admission then measurement. Attempting to measure an unadmitted candidate packet is strictly refused.

In [ ]:
carrier = opa.SimplicialCarrier(vertices=["v0", "v1", "v2"], edges=[("v0", "v1"), ("v1", "v2"), ("v0", "v2")], faces=[("v0", "v1", "v2")])
h_exact = {("v0", "v1"): Fraction(1), ("v1", "v2"): Fraction(1), ("v0", "v2"): Fraction(2)}

# Measurement with admitted=True succeeds
sig = opa.ObstructionCalculus.analyze_1cochain(carrier, h_exact, admitted=True)
print(f"Phi_1: {sig.phi_1}, Gamma_2: {sig.gamma_2}, Status: {sig.status_label}")
assert sig.phi_1 == Fraction(0) and sig.gamma_2 == Fraction(0)
assert sig.status_label == "REMOVABLE_GAUGE"

## 2. Seven-Step Local Declaration Order (OE001-02)

Enforcing sequential declaration validity: Grounding -> Carrier -> Observer -> Primitive Difference -> ZeroRel -> Witness -> Quotient.

In [ ]:
obs = opa.ObserverPacket(observer_id="obs_triad", gauge_name="L1", resolution=Fraction(1, 1000))
decl = opa.LocalDeclarationOrder(
    step0_grounding_hash="sha256:genesis_artifact_001",
    step1_carrier=carrier,
    step2_observer=obs,
    step3_primitive_difference=h_exact,
    step4_zero_rel_rule="EXACT_GAUGE_BOUNDARY",
    step5_witness_certificate="WITNESS_CERT_DUAL_BOUND",
    step6_quotient_policy="Q1_SEPARATED"
)
ok, reason = decl.validate_declaration_order()
print(f"Declaration order valid: {ok}, reason: {reason}")
assert ok and reason == "ACCEPT_DECLARATION"

## 3. Prime 002: CanContinue Reconciliation and Bounded Termination

The 9 canonical conjuncts evaluate continuation and reduce to 3 summary conditions.

In [ ]:
engine = opa.PrimeTriadEngine(budget=30, delta_control=10)
can_cont = engine.can_continue(
    0,
    has_obligation=True,
    has_witness=True,
    next_carrier_declared=True,
    adapter_sound=True,
    detector_adequate=True,
    minimal_lift=True,
    skeleton_preserved=True,
    budget_remains=True,
    gtla_refusal=False
)
print("CanContinue:", can_cont)
assert can_cont is True

# Step execution bounded by initial budget
s1 = engine.execute_step()
s2 = engine.execute_step()
s3 = engine.execute_step()
s4 = engine.execute_step()
print("Steps:", [s1, s2, s3, s4], "ops:", engine.ops_count)
assert s1 == s2 == s3 == "STEP_EXECUTED"
assert s4 == "TERMINAL_WITH_DEBT"
assert engine.ops_count == 3 == 30 // 10

## 4. Scoped Terminalization and Merkle Trace Tape

In [ ]:
terminalizer = opa.DDATerminalizer(engine, genesis_hash="GENESIS_ROOT")
terminalizer.process_step("E", {"action": "exact_repair"})
terminalizer.process_step("R", {"action": "residual_projection"})
ok_tape, tape_status = opa.verify_merkle_tape(terminalizer.merkle_tape, "GENESIS_ROOT")
print(f"Tape verified: {ok_tape} ({tape_status})")
assert ok_tape is True